# 13 — Produtos TOTVS e termos principais

Este módulo transforma a transcrição em candidatos fundamentados na base TOTVS e em até dez termos úteis para a revisão comercial. Quando o modelo e o índice local existem, usa E5; caso contrário, mantém BM25 com aliases como fallback auditável.

## Artefatos opcionais do E5

O índice `rag_multilingual_e5_small_embeddings.npz` é produzido pelo notebook 10 e não é versionado. O modelo e o índice são carregados somente na primeira consulta do modo `auto` ou `full`.

In [ ]:
E5_MODEL_NAME = "intfloat/multilingual-e5-small"
E5_EMBEDDINGS_PATH = "data/processed/rag_multilingual_e5_small_embeddings.npz"
_E5_RETRIEVER = None
_E5_LOAD_ERROR = None


## Representação lexical

Campos mais específicos — produto, título e palavras-chave — recebem mais peso. Os aliases reconhecem como equivalentes formas como `Protheus` e `TOTVS Protheus`.

In [ ]:
def _document_tokens(document: dict[str, Any]) -> list[str]:
    weighted_fields = [
        (document.get("product", ""), 4),
        (document.get("title", ""), 3),
        (" ".join(document.get("keywords", [])), 3),
        (" ".join(document.get("competitors", [])), 2),
        (document.get("category", ""), 1),
        (" ".join(document.get("segments", [])), 1),
        (" ".join(document.get("related_products", [])), 1),
        (document.get("content", ""), 1),
    ]
    return [token for text, weight in weighted_fields for token in _tokens(text) * weight]

def _query_tokens(transcription: str, alias_groups: list[dict[str, Any]]) -> tuple[list[str], set[str]]:
    normalized = f" {_normalize(transcription)} "
    query = _tokens(transcription)
    explicit_products: set[str] = set()
    for group in alias_groups:
        variants = [group["canonical"], *group.get("aliases", [])]
        if any(f" {_normalize(variant)} " in normalized for variant in variants):
            canonical = group["canonical"]
            explicit_products.add(_normalize(canonical))
            query.extend(_tokens(canonical) * 3)
    return query, explicit_products


## Evidências compartilhadas pelos rankings

BM25 e E5 calculam relevância de formas diferentes, mas entregam o mesmo contrato. Estas funções concentram a agregação das fontes e uma descrição curta de cada documento, evitando que os dois caminhos produzam estruturas divergentes.

In [ ]:
def _add_product_evidence(
    grouped: dict[str, dict[str, Any]],
    product: str,
    raw_score: float,
    explicit_match: bool,
    matched_terms: set[str],
    document: dict[str, Any],
    source_urls: list[str],
) -> None:
    candidate = grouped.setdefault(
        product,
        {
            "raw_score": float("-inf"),
            "explicit_match": False,
            "matched_terms": set(),
            "documents": {},
            "sources": [],
        },
    )
    candidate["raw_score"] = max(candidate["raw_score"], raw_score)
    candidate["explicit_match"] = candidate["explicit_match"] or explicit_match
    candidate["matched_terms"].update(matched_terms)
    candidate["documents"][document["id"]] = {
        "id": document["id"],
        "title": document.get("title"),
        "document_type": document.get("document_type"),
        "product": document.get("product"),
    }
    candidate["sources"].extend(source_urls)

def _serialize_product_candidates(
    grouped: dict[str, dict[str, Any]],
    top_k: int,
    *,
    score_type: str,
    engine: str,
    model: str | None,
    score_transform: Any,
) -> list[dict[str, Any]]:
    ranked = sorted(grouped.items(), key=lambda item: (-item[1]["raw_score"], item[0]))[:top_k]
    return [
        {
            "product": product,
            "score": round(min(max(score_transform(values["raw_score"]), 0.0), 1.0), 6),
            "score_type": score_type,
            "engine": engine,
            "model": model,
            "explicit_match": values["explicit_match"],
            "matched_terms": sorted(values["matched_terms"])[:10],
            "document_ids": list(values["documents"]),
            "documents": list(values["documents"].values()),
            "sources": list(dict.fromkeys(values["sources"])),
        }
        for product, values in ranked
    ]


## Ranking com fontes

O cálculo BM25 consolida documentos pelo produto, exige ao menos uma URL de fonte e conserva no máximo três candidatos. `explicit_match` distingue uma citação direta de uma correspondência apenas contextual.

In [ ]:
def _rank_products_bm25(transcription: str, top_k: int = 3) -> list[dict[str, Any]]:
    knowledge_base, alias_groups = _load_catalog()
    query, explicit_products = _query_tokens(transcription, alias_groups)
    if not query:
        return []

    document_tokens = [_document_tokens(document) for document in knowledge_base]
    frequencies = [Counter(tokens) for tokens in document_tokens]
    lengths = [len(tokens) for tokens in document_tokens]
    average_length = sum(lengths) / max(len(lengths), 1)
    document_frequency = Counter()
    for tokens in document_tokens:
        document_frequency.update(set(tokens))

    grouped: dict[str, dict[str, Any]] = {}
    for index, (document, tokens, term_frequency, length) in enumerate(
        zip(knowledge_base, document_tokens, frequencies, lengths)
    ):
        score = 0.0
        for term in query:
            frequency = term_frequency.get(term, 0)
            if not frequency:
                continue
            seen_in = document_frequency[term]
            inverse_document_frequency = math.log(
                1 + (len(knowledge_base) - seen_in + 0.5) / (seen_in + 0.5)
            )
            denominator = frequency + 1.5 * (1 - 0.75 + 0.75 * length / average_length)
            score += inverse_document_frequency * frequency * 2.5 / denominator

        product = document.get("product") or document.get("title")
        normalized_product = _normalize(product)
        explicit_match = any(
            alias in normalized_product
            or normalized_product in alias
            or set(_tokens(alias)).issubset(set(_tokens(product)))
            for alias in explicit_products
        )
        if explicit_match:
            score += 8.0
        if score < 2.0:
            continue

        source_urls = [source["url"] for source in document.get("sources", []) if source.get("url")]
        if not source_urls:
            continue
        matched = set(query) & set(tokens)
        _add_product_evidence(
            grouped, product, score, explicit_match, matched, document, source_urls
        )

    if not grouped:
        return []
    highest_score = max(values["raw_score"] for values in grouped.values())
    return _serialize_product_candidates(
        grouped,
        top_k,
        score_type="heuristic",
        engine="bm25_aliases",
        model=None,
        score_transform=lambda raw_score: raw_score / highest_score,
    )


## Formato comum dos candidatos E5

A similaridade de cosseno é normalizada para o intervalo de zero a um somente para facilitar ordenação. Ela continua identificada como similaridade, não como probabilidade. Documentos sem fonte são descartados.

In [ ]:
def _format_e5_candidates(
    transcription: str, scores: list[float], top_k: int
) -> list[dict[str, Any]]:
    knowledge_base, alias_groups = _load_catalog()
    query, explicit_products = _query_tokens(transcription, alias_groups)
    grouped: dict[str, dict[str, Any]] = {}
    ranked_indices = sorted(range(len(scores)), key=lambda index: -scores[index])
    for index in ranked_indices:
        raw_score = float(scores[index])
        if raw_score < 0.20:
            continue
        document = knowledge_base[index]
        source_urls = [
            source["url"]
            for source in document.get("sources", [])
            if source.get("url")
        ]
        if not source_urls:
            continue
        product = document.get("product") or document.get("title")
        product_tokens = set(_tokens(product))
        explicit_match = any(
            set(_tokens(alias)).issubset(product_tokens) for alias in explicit_products
        )
        matched = set(query) & set(_document_tokens(document))
        _add_product_evidence(
            grouped, product, raw_score, explicit_match, matched, document, source_urls
        )

    return _serialize_product_candidates(
        grouped,
        top_k,
        score_type="normalized_cosine_similarity",
        engine="multilingual_e5_small",
        model=E5_MODEL_NAME,
        score_transform=lambda raw_score: (raw_score + 1.0) / 2.0,
    )


## Carregamento e roteamento do retriever

O loader confere se os IDs do índice continuam na mesma ordem da base TOTVS, detecta GPU e cria um adaptador de consulta. `auto` registra o motivo do fallback; `full` exige E5; `fallback` usa BM25 diretamente.

In [ ]:
def _load_e5_retriever():
    global _E5_RETRIEVER, _E5_LOAD_ERROR
    if _E5_RETRIEVER is not None:
        return _E5_RETRIEVER
    if _E5_LOAD_ERROR is not None:
        raise RuntimeError("Retriever E5 indisponível.") from _E5_LOAD_ERROR
    try:
        import numpy as np
        import torch
        import torch.nn.functional as functional
        from transformers import AutoModel, AutoTokenizer

        embedding_path = _project_root() / E5_EMBEDDINGS_PATH
        if not embedding_path.is_file():
            raise FileNotFoundError(f"Índice E5 não encontrado: {embedding_path}")
        knowledge_base, _ = _load_catalog()
        saved = np.load(embedding_path, allow_pickle=False)
        expected_ids = [document["id"] for document in knowledge_base]
        if saved["document_ids"].tolist() != expected_ids:
            raise ValueError("O índice E5 não corresponde à versão atual da base TOTVS.")
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        tokenizer = AutoTokenizer.from_pretrained(E5_MODEL_NAME)
        model = AutoModel.from_pretrained(E5_MODEL_NAME).to(device).eval()
        document_embeddings = torch.as_tensor(
            saved["embeddings"], dtype=torch.float32, device=device
        )

        def retrieve(text: str, top_k: int = 3) -> list[dict[str, Any]]:
            encoded = tokenizer(
                ["query: " + text],
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt",
            )
            encoded = {name: value.to(device) for name, value in encoded.items()}
            with torch.inference_mode():
                hidden = model(**encoded).last_hidden_state.float()
            mask = encoded["attention_mask"].unsqueeze(-1)
            pooled = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1)
            query_embedding = functional.normalize(pooled, p=2, dim=1)
            scores = (query_embedding @ document_embeddings.T)[0].cpu().tolist()
            return _format_e5_candidates(text, scores, top_k)

        _E5_RETRIEVER = retrieve
        return _E5_RETRIEVER
    except Exception as error:
        _E5_LOAD_ERROR = error
        raise RuntimeError("Retriever E5 indisponível.") from error


In [ ]:
def _analyze_products(
    transcription: str, mode: str
) -> tuple[list[dict[str, Any]], str, dict[str, str] | None]:
    if mode == "fallback":
        return _rank_products_bm25(transcription), "fallback", None
    try:
        retriever = _load_e5_retriever()
    except Exception as error:
        if mode == "full":
            raise RuntimeError("O modo full exige o modelo e o índice E5.") from error
        return _rank_products_bm25(transcription), "fallback", _fallback_reason(error)
    return retriever(transcription, top_k=3), "model", None


## Proteção durante a extração de termos

E-mail, telefone e CPF são removidos somente da cópia usada para extrair termos. Essa filtragem não altera `transcricao_original`.

In [ ]:
def _text_without_personal_data(text: str) -> str:
    sanitized = re.sub(r"\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b", " ", text, flags=re.I)
    sanitized = re.sub(r"\b\d{3}\.?\d{3}\.?\d{3}-?\d{2}\b", " ", sanitized)
    sanitized = re.sub(
        r"(?<!\d)(?:\+?55\s*)?(?:\(?\d{2}\)?[\s.-]*)?9?\d{4}[\s.-]*\d{4}(?!\d)",
        " ",
        sanitized,
    )
    return sanitized


## Priorização dos termos

Produtos citados vêm primeiro, seguidos de dores, concorrentes e vocabulário comercial. A frequência geral completa a lista somente quando ainda há espaço, sempre com limite de dez itens únicos.

In [ ]:
def _extract_key_terms(transcription: str, limit: int = 10) -> list[str]:
    sanitized = _text_without_personal_data(transcription)
    normalized = _normalize(sanitized)
    knowledge_base, alias_groups = _load_catalog()
    prioritized: list[str] = []
    seen: set[str] = set()

    def add(term: str) -> None:
        canonical = _normalize(term).strip()
        if not canonical or canonical in seen or len(prioritized) >= limit:
            return
        prioritized.append(term)
        seen.add(canonical)

    product_terms: list[tuple[int, str]] = []
    for group in alias_groups:
        variants = [group["canonical"], *group.get("aliases", [])]
        positions = [
            normalized.find(_normalize(variant))
            for variant in variants
            if re.search(rf"(?<!\w){re.escape(_normalize(variant))}(?!\w)", normalized)
        ]
        if positions:
            product_terms.append((min(positions), group["canonical"]))
    for _, term in sorted(product_terms):
        add(term)

    pain_hits = _signal_hits(normalized, OPPORTUNITY_PAIN_SIGNALS)
    for term in sorted(pain_hits, key=lambda item: (normalized.find(item), item)):
        add(term)

    competitors = {
        competitor
        for document in knowledge_base
        for competitor in document.get("competitors", [])
        if competitor
    }
    competitor_hits = [
        competitor
        for competitor in competitors
        if re.search(
            rf"(?<!\w){re.escape(_normalize(competitor))}(?!\w)", normalized
        )
    ]
    for term in sorted(competitor_hits, key=lambda item: (normalized.find(_normalize(item)), item)):
        add(term)

    commercial_signals = (
        OPPORTUNITY_INTENT_SIGNALS
        | OPPORTUNITY_BUY_SIGNALS
        | set(HIGH_CHURN_SIGNALS)
        | set(MEDIUM_CHURN_SIGNALS)
    )
    commercial_hits = _signal_hits(normalized, commercial_signals)
    for term in sorted(commercial_hits, key=lambda item: (normalized.find(item), item)):
        add(term)

    token_counts = Counter(_tokens(sanitized))
    first_position = {token: normalized.find(token) for token in token_counts}
    for token, _ in sorted(
        token_counts.items(),
        key=lambda item: (-item[1], first_position[item[0]], item[0]),
    ):
        if token.isdigit() or any(
            re.search(rf"(?<!\w){re.escape(token)}(?!\w)", existing)
            for existing in seen
        ):
            continue
        add(token)
    return prioritized
